# 04 Build Gold Dashboard Tables

Build Power BI-ready Gold tables from Silver data: current snapshot, 5-minute facts, 30-minute and daily aggregates, price events, KPIs, data freshness, and optional interconnector summaries.

## Configure Gold Run

This cell detects local versus Fabric runtime and defines run parameters. Spark transformations run only in Fabric.

In [1]:
# Cell purpose: Configure Gold Build Run.
from pathlib import Path
from typing import Any, cast
import importlib
import os
import sys
import uuid

try:
    spark
    is_local_run = False
except NameError:
    is_local_run = True

run_id = str(uuid.uuid4())
high_price_threshold_aud_mwh = 300.0
extreme_price_threshold_aud_mwh = 1000.0

print(f"run_id={run_id}")
print(f"runtime={'local' if is_local_run else 'fabric'}")


run_id=a430e180-e0e7-4fa4-be2b-096b70bc88ef
runtime=local


## Resolve Runtime Paths and Imports

This cell resolves package paths and imports Spark helpers only in Fabric.

In [2]:
# Cell purpose: Resolve Runtime Paths and Imports.
def find_repo_root(start: Path) -> Path:
    """Find the local repo root from a notebook working directory."""

    for candidate in (start, *start.parents):
        if (candidate / "src" / "nem_fabric").exists():
            return candidate
    return start


package_paths = []
# Cell purpose: Load Silver price and demand source table.
if is_local_run:
    repo_root = find_repo_root(Path.cwd())
    local_output_root = repo_root / "data"
    local_tables_root = local_output_root / "tables"
    package_paths.append(repo_root / "src")
else:
    package_paths.append(
        Path(os.getenv("FABRIC_NOTEBOOK_LIB_PATH", "/lakehouse/default/Files/libs"))
    )

for package_path in package_paths:
    if package_path.exists() and str(package_path) not in sys.path:
        sys.path.insert(0, str(package_path))

# Cell purpose: Build 5-minute regional Gold fact table.
if is_local_run:
    import pandas as pd

    from nem_fabric.common_transformations import (
        build_30min_region_aggregation,
        build_current_snapshot,
        build_daily_region_summary,
        build_dashboard_kpis,
        build_dashboard_supply_demand_components,
        build_data_freshness,
        build_gold_interconnector_flows,
        build_gold_region_5min,
        build_price_spikes,
    )

    def write_local_table(table_name: str, frame: pd.DataFrame) -> None:
        """Overwrite a local CSV table for deterministic local Gold reruns."""

        local_tables_root.mkdir(parents=True, exist_ok=True)
        frame.to_csv(local_tables_root / f"{table_name}.csv", index=False)
else:
    F = importlib.import_module("pyspark.sql.functions")
    Window = importlib.import_module("pyspark.sql.window").Window

print(
    "Python search paths added:", [str(path) for path in package_paths if path.exists()]
)


Python search paths added: ['c:\\Users\\brcol\\My Drive\\Documents\\!!!Resume\\Sample Work\\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\\src']


## Load Silver Source

This cell confirms the required Silver price/demand table exists and loads it as the base for all mandatory Gold tables.

In [3]:
# Cell purpose: Load Silver price and demand source table.
if is_local_run:
    silver_path = local_tables_root / "nem_silver_price_demand_5min.csv"
    if not silver_path.exists():
        raise RuntimeError(
            "nem_silver_price_demand_5min.csv does not exist. Run notebook 03 first."
        )
    silver = pd.read_csv(silver_path)
    print(f"Local Silver rows loaded: {len(silver)}")
else:
    # Cell purpose: Load Silver Source.
    def table_exists(table_name: str) -> bool:
        """Return True when a Lakehouse table exists in the current Spark catalogue."""
        return spark.catalog.tableExists(table_name)

    if not table_exists("nem_silver_price_demand_5min"):
        raise RuntimeError(
            "nem_silver_price_demand_5min does not exist. Run notebook 03 first."
        )

    silver = spark.table("nem_silver_price_demand_5min")


Local Silver rows loaded: 2130


## Build Five-Minute Regional Fact

This cell creates the main Power BI fact table with clean column names, price bands, event flags, and one-hour rolling averages.

In [4]:
# Cell purpose: Build 5-minute regional Gold fact table.
if is_local_run:
    gold_5min = build_gold_region_5min(silver, run_id=run_id)
    write_local_table("nem_gold_region_5min", gold_5min)
    print(f"Local Gold 5-minute rows written: {len(gold_5min)}")
else:
    # Main 5-minute Gold fact. Flags and bands are precomputed so Power BI stays simple.
    rolling_window = (
        Window.partitionBy("region")
        .orderBy(F.col("settlement_datetime").cast("long"))
        .rangeBetween(-3600, 0)
    )

    spark_silver = cast(Any, silver)
    gold_5min = (
        spark_silver.withColumn(
            "price_band",
            F.when(F.col("price_aud_mwh") < 0, "Negative")
            .when(F.col("price_aud_mwh") >= extreme_price_threshold_aud_mwh, "Extreme")
            .when(F.col("price_aud_mwh") >= high_price_threshold_aud_mwh, "High")
            .otherwise("Normal"),
        )
        .withColumn("is_negative_price", F.col("price_aud_mwh") < 0)
        .withColumn(
            "is_high_price", F.col("price_aud_mwh") >= high_price_threshold_aud_mwh
        )
        .withColumn(
            "is_extreme_price",
            F.col("price_aud_mwh") >= extreme_price_threshold_aud_mwh,
        )
        .withColumn("rolling_avg_price_1h", F.avg("price_aud_mwh").over(rolling_window))
        .withColumn("rolling_avg_demand_1h", F.avg("demand_mw").over(rolling_window))
        .withColumn("gold_loaded_datetime", F.current_timestamp())
        .withColumn("run_id", F.lit(run_id))
        .select(
            "settlement_datetime",
            "trading_date",
            "year",
            "month",
            "day",
            "interval_hour",
            "interval_minute",
            "region",
            "region_name",
            "intervention",
            "price_aud_mwh",
            "demand_mw",
            "available_generation_mw",
            "available_load_mw",
            "demand_forecast_mw",
            "dispatchable_generation_mw",
            "dispatchable_load_mw",
            "net_interchange_mw",
            "excess_generation_mw",
            "dashboard_demand_mw",
            "semi_scheduled_generation_mw",
            "scheduled_generation_mw",
            "dashboard_generation_mw",
            "price_band",
            "is_negative_price",
            "is_high_price",
            "is_extreme_price",
            "rolling_avg_price_1h",
            "rolling_avg_demand_1h",
            "gold_loaded_datetime",
            "run_id",
        )
    )

    gold_5min.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).partitionBy("trading_date").saveAsTable("nem_gold_region_5min")


Local Gold 5-minute rows written: 2130


## Build Current Snapshot

This cell selects the latest interval for each region to support overview KPI cards and current market status visuals.

In [5]:
# Cell purpose: Build current regional dashboard snapshot.
if is_local_run:
    snapshot = build_current_snapshot(gold_5min)
    write_local_table("nem_gold_dashboard_current_snapshot", snapshot)
    print(f"Local current snapshot rows written: {len(snapshot)}")
else:
    # Current snapshot: latest interval by region for KPI cards.
    latest_by_region = Window.partitionBy("region").orderBy(
        F.col("settlement_datetime").desc()
    )
    spark_gold_5min = cast(Any, gold_5min)
    snapshot = (
        spark_gold_5min.withColumn("row_number", F.row_number().over(latest_by_region))
        .filter(F.col("row_number") == 1)
        .drop("row_number")
    )
    snapshot.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable("nem_gold_dashboard_current_snapshot")
    display(cast(Any, snapshot).orderBy("region"))


Local current snapshot rows written: 5


## Build Current Supply/Demand Components

This cell reshapes the current snapshot into demand and generation components for Power BI stacked bar visuals.

In [6]:
# Cell purpose: Build current dashboard supply/demand component table.
if is_local_run:
    supply_demand_components = build_dashboard_supply_demand_components(snapshot)
    write_local_table(
        "nem_gold_dashboard_supply_demand_components", supply_demand_components
    )
    print(
        "Local supply/demand component rows written: "
        f"{len(supply_demand_components)}"
    )
else:
    # Long-format current table supports one demand row and a stacked generation row per region.
    demand_components = snapshot.select(
        "settlement_datetime",
        "trading_date",
        "region",
        "region_name",
        F.lit("Demand").alias("metric_group"),
        F.lit("Demand").alias("component"),
        F.lit(1).alias("component_sort_order"),
        F.col("dashboard_demand_mw").alias("value_mw"),
        "gold_loaded_datetime",
        "run_id",
    )
    scheduled_components = snapshot.select(
        "settlement_datetime",
        "trading_date",
        "region",
        "region_name",
        F.lit("Generation").alias("metric_group"),
        F.lit("Scheduled Generation").alias("component"),
        F.lit(1).alias("component_sort_order"),
        F.col("scheduled_generation_mw").alias("value_mw"),
        "gold_loaded_datetime",
        "run_id",
    )
    semi_scheduled_components = snapshot.select(
        "settlement_datetime",
        "trading_date",
        "region",
        "region_name",
        F.lit("Generation").alias("metric_group"),
        F.lit("Semi-scheduled Generation").alias("component"),
        F.lit(2).alias("component_sort_order"),
        F.col("semi_scheduled_generation_mw").alias("value_mw"),
        "gold_loaded_datetime",
        "run_id",
    )
    supply_demand_components = demand_components.unionByName(
        scheduled_components
    ).unionByName(semi_scheduled_components)
    supply_demand_components.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable("nem_gold_dashboard_supply_demand_components")
    display(
        cast(Any, supply_demand_components).orderBy(
            "region", "metric_group", "component_sort_order"
        )
    )


Local supply/demand component rows written: 15


## Build Thirty-Minute Regional Aggregate

This cell aggregates 5-minute intervals to 30-minute regional grain for dashboard users who want a less granular view.

In [7]:
# Cell purpose: Build 30-minute regional Gold aggregate.
if is_local_run:
    gold_30min = build_30min_region_aggregation(gold_5min)
    write_local_table("nem_gold_region_30min", gold_30min)
    print(f"Local 30-minute Gold rows written: {len(gold_30min)}")
else:
    # 30-minute Gold aggregate for interval-granularity switching in Power BI.
    spark_gold_5min = cast(Any, gold_5min)
    gold_30min = (
        spark_gold_5min.withColumn(
            "settlement_30min", F.window("settlement_datetime", "30 minutes").start
        )
        .groupBy("region", "region_name", "settlement_30min")
        .agg(
            F.avg("price_aud_mwh").alias("price_aud_mwh"),
            F.avg("demand_mw").alias("demand_mw"),
            F.max("price_aud_mwh").alias("max_price_aud_mwh"),
            F.min("price_aud_mwh").alias("min_price_aud_mwh"),
            F.sum(F.col("is_high_price").cast("int")).alias(
                "high_price_interval_count"
            ),
            F.sum(F.col("is_negative_price").cast("int")).alias(
                "negative_price_interval_count"
            ),
        )
        .withColumn("trading_date", F.to_date("settlement_30min"))
        .withColumn("interval_hour", F.hour("settlement_30min"))
        .withColumn("interval_minute", F.minute("settlement_30min"))
    )
    gold_30min.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).partitionBy("trading_date").saveAsTable("nem_gold_region_30min")


Local 30-minute Gold rows written: 190


## Build Daily Regional Summary

This cell calculates daily average, maximum, minimum, volatility, demand, and event-count metrics by region.

In [8]:
# Cell purpose: Build daily regional Gold summary.
if is_local_run:
    gold_daily = build_daily_region_summary(gold_5min)
    write_local_table("nem_gold_region_daily", gold_daily)
    print(f"Local daily Gold rows written: {len(gold_daily)}")
else:
    # Daily regional summary for trend and volatility pages.
    spark_gold_5min = cast(Any, gold_5min)
    gold_daily = spark_gold_5min.groupBy("region", "region_name", "trading_date").agg(
        F.avg("price_aud_mwh").alias("daily_avg_price"),
        F.max("price_aud_mwh").alias("daily_max_price"),
        F.min("price_aud_mwh").alias("daily_min_price"),
        F.stddev_pop("price_aud_mwh").alias("daily_price_volatility"),
        F.avg("demand_mw").alias("daily_avg_demand"),
        F.max("demand_mw").alias("daily_max_demand"),
        F.sum(F.col("is_high_price").cast("int")).alias("high_price_interval_count"),
        F.sum(F.col("is_extreme_price").cast("int")).alias(
            "extreme_price_interval_count"
        ),
        F.sum(F.col("is_negative_price").cast("int")).alias(
            "negative_price_interval_count"
        ),
    )
    gold_daily.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).partitionBy("trading_date").saveAsTable("nem_gold_region_daily")


Local daily Gold rows written: 10


## Build Price Event Table

This cell creates a drill-through table for high, extreme, and negative price intervals.

In [9]:
# Cell purpose: Build price spike and negative price event table.
if is_local_run:
    price_spikes = build_price_spikes(gold_5min)
    write_local_table("nem_gold_price_spikes", price_spikes)
    print(f"Local price event rows written: {len(price_spikes)}")
else:
    # Price events table supports detailed drill-through for spikes and negative intervals.
    spark_gold_5min = cast(Any, gold_5min)
    price_spikes = spark_gold_5min.filter(F.col("is_high_price") | F.col("is_negative_price"))
    price_spikes.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).partitionBy("trading_date").saveAsTable("nem_gold_price_spikes")


Local price event rows written: 837


## Build KPIs and Freshness

This cell creates dashboard-level KPI and data freshness tables so Power BI can show operational status without extra transformations.

In [10]:
# Cell purpose: Build dashboard KPI and data freshness tables.
if is_local_run:
    kpis = build_dashboard_kpis(snapshot, run_id=run_id)
    freshness = build_data_freshness(kpis)
    write_local_table("nem_gold_dashboard_kpis", kpis)
    write_local_table("nem_gold_data_freshness", freshness)
    print("Local KPI and freshness Gold tables written.")
else:
    # Dashboard KPIs and freshness tables make operational status visible in Power BI.
    spark_snapshot = cast(Any, snapshot)
    kpis = (
        spark_snapshot.groupBy()
        .agg(
            F.max("settlement_datetime").alias("latest_settlement_datetime"),
            F.avg("price_aud_mwh").alias("avg_current_price_aud_mwh"),
            F.sum("demand_mw").alias("current_total_demand_mw"),
            F.countDistinct("region").alias("regions_available"),
        )
        .withColumn("run_id", F.lit(run_id))
        .withColumn("gold_loaded_datetime", F.current_timestamp())
    )
    kpis.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable("nem_gold_dashboard_kpis")

    freshness = (
        kpis.select("latest_settlement_datetime", "gold_loaded_datetime", "run_id")
        .withColumn(
            "freshness_minutes",
            (
                F.unix_timestamp(F.current_timestamp())
                - F.unix_timestamp("latest_settlement_datetime")
            )
            / 60.0,
        )
        .withColumn(
            "status",
            F.when(F.col("freshness_minutes") <= 15, "Fresh")
            .when(F.col("freshness_minutes") <= 60, "Delayed")
            .otherwise("Stale"),
        )
        .withColumnRenamed("gold_loaded_datetime", "last_successful_ingestion_datetime")
    )
    freshness.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable("nem_gold_data_freshness")
    display(cast(Any, kpis))


Local KPI and freshness Gold tables written.


## Build Optional Interconnector Gold

This cell creates a Power BI-ready interconnector flow table when the Silver interconnector source is available.

In [11]:
# Cell purpose: Build optional interconnector Gold table.
if is_local_run:
    interconnector_path = local_tables_root / "nem_silver_interconnector_flows.csv"
    if interconnector_path.exists():
        interconnector = pd.read_csv(interconnector_path)
        interconnector_gold = build_gold_interconnector_flows(interconnector)
        write_local_table("nem_gold_interconnector_flows_5min", interconnector_gold)
        print(f"Local interconnector Gold rows written: {len(interconnector_gold)}")
    else:
        print("Skipping interconnector Gold table because local Silver source is unavailable.")
else:
    # Optional interconnector Gold table. Built only when notebook 03 created the Silver source.
    if table_exists("nem_silver_interconnector_flows"):
        interconnector = spark.table("nem_silver_interconnector_flows")
        spark_interconnector = cast(Any, interconnector)
        interconnector_gold = (
            spark_interconnector.withColumn(
                "flow_direction",
                F.when(F.col("flow_mw") >= 0, "Forward").otherwise("Reverse"),
            )
            .withColumn("interval_hour", F.hour("settlement_datetime"))
            .withColumn("interval_minute", F.minute("settlement_datetime"))
        )
        interconnector_gold.write.format("delta").mode("overwrite").option(
            "overwriteSchema", "true"
        ).partitionBy("trading_date").saveAsTable("nem_gold_interconnector_flows_5min")
    else:
        print(
            "Skipping interconnector Gold table because Silver source is unavailable."
        )


Local interconnector Gold rows written: 2556
